# Typed language interop — Usage and Reference
JupyterSharp C# embedding of PySharp and MatSharp, using NuGet assemblies. Python closes a host-injected CLR generic type; MATLAB solves a typed system through its registered adapter. Neither path passes Decimal128 through double. This is embedding, not proof of a native PySharp/MatSharp JupyterSharp plugin.
Run `dotnet run --file ops/RunEcosystemPilot.cs` to restore packages, then `dotnet run --file ops/RunLanguageInteropPilot.cs`. Underdetermined typed MATLAB systems, NumPy typed arrays and generic inference remain unsupported.

In [ ]:
#r "D:/Dev/BuildArtifacts/net11-pack/consumer-cache/pysharp.interpreter/2.6.1-net11-preview.2/lib/net11.0/PySharpLib.dll"
#r "D:/Dev/BuildArtifacts/net11-pack/consumer-cache/matsharp.interpreter/0.1.1-net11-preview.3/lib/net11.0/MatSharpLib.dll"
#r "D:/Dev/BuildArtifacts/net11-pack/consumer-cache/matsharp.ndsharp/0.1.1-net11-preview.3/lib/net11.0/MatSharpLib.NDSharp.dll"
#r "D:/Dev/BuildArtifacts/net11-pack/consumer-cache/mathemasharp.linearalgebra/0.2.1-net11-preview.1/lib/net11.0/MathemaSharp.LinearAlgebra.dll"
#r "D:/Dev/BuildArtifacts/net11-pack/consumer-cache/ndsharp/2.8.1-net11-preview.1/lib/net11.0/NDSharp.dll"
using System.Numerics;
using System.Collections.Generic;
using System.IO;


In [ ]:
var output = new StringWriter();
var python = new PySharpLib.PyEngine(output);
python.SetVariable("ClrList", typeof(List<>)); python.SetVariable("D128", typeof(Decimal128));
python.Run("from mathema import decimal128\nList128 = ClrList[D128]\nvalues = List128()\nvalues.Add(decimal128('1.0000000000000000000000001'))\nprint(values[0] - 1 == decimal128('1e-25'))");
if (output.ToString() != "True\n") throw new Exception("Python generic type lost precision");
Console.WriteLine("PYSHARP_GENERIC_TYPE_PRECISION_OK");

PYSHARP_GENERIC_TYPE_PRECISION_OK

In [ ]:
var matlab = new MatSharpLib.MatEngine();
MatSharpLib.NDSharpBridge.NDSharpFunctions.Register(matlab);
matlab.Run("a=[decimal128(1) decimal128(0);decimal128(0) decimal128(1)]; b=[decimal128('1.0000000000000000000000001');decimal128(2)]; x=a\\b;");
var increment = (MatSharpLib.Runtime.GenericScalar<Decimal128>)matlab.Eval("x(1)-decimal128(1)");
if (increment.Value != Decimal128.Parse("1e-25")) throw new Exception("MATLAB solve lost precision");
Console.WriteLine("MATSHARP_TYPED_SOLVE_PRECISION_OK");

MATSHARP_TYPED_SOLVE_PRECISION_OK